# Localizar los modelos de las cinco semillas

Este cuaderno busca los pesos guardados de BERT, DistilBERT y RoBERTuito para México, España y Cuba. Distingue modelos y checkpoints de los CSV de predicciones. **No entrena, no necesita GPU y no carga ni modifica los modelos.**

Ejecutar todas las celdas en la máquina donde se entrenaron los modelos. No necesitas saber la ruta exacta: busca en el proyecto, Documentos, Descargas, Escritorio, OneDrive y otras unidades de disco locales en Windows. Omite carpetas de sistema y entornos de Python. Las carpetas sin acceso se registran.

La presencia de un archivo de pesos no garantiza que sea el modelo final: el informe distingue exports con metadatos, checkpoints del test final y otros candidatos. Los archivos comprimidos se enumeran como posibles respaldos y deben inspeccionarse aparte.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ_PROYECTO = next((p for p in candidatas if (p / "data" / "resultados_robertuito_grouped_5x3_tpe20").is_dir()), None)
if RAIZ_PROYECTO is None:
    raise FileNotFoundError("Abre el cuaderno desde la carpeta notebooks o la raíz del proyecto.")
SALIDA = RAIZ_PROYECTO / "data" / "localizacion_pesos_transformers" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SALIDA.mkdir(parents=True, exist_ok=False)


In [ ]:
"""Localiza pesos y sus metadatos. Solo lectura; no carga modelos ni ejecuta entrenamiento."""
from pathlib import Path
import csv
import json
import os
import pickletools
import re
import zipfile

OMITIR = {'.git','.venv','venv','env','.uv-cache','.uv-python','node_modules','site-packages',
    '__pycache__','.cache','Cache','Caches','$Recycle.Bin','System Volume Information',
    'Windows','Program Files','Program Files (x86)','ProgramData','AppData'}
MODELOS = ['BERT','DistilBERT','RoBERTuito']
VARIANTES = ['mx','es','cu']
SEMILLAS = [42,52,62,72,82]


def raices_automaticas(proyecto):
    candidatas=[Path(proyecto)]
    home=Path.home()
    candidatas += [home/n for n in ['Documents','Downloads','Desktop','OneDrive']]
    if os.name == 'nt':
        import ctypes
        unidad_sistema=Path.home().drive.upper()
        for letra in 'CDEFGHIJKLMNOPQRSTUVWXYZ':
            raiz=letra+':\\'
            if (letra+':').upper()!=unidad_sistema and ctypes.windll.kernel32.GetDriveTypeW(raiz) in [2,3]:
                candidatas.append(Path(raiz))
    elif Path('/Volumes').is_dir():
        candidatas += [p for p in Path('/Volumes').iterdir() if p.is_dir() and p.name!='Macintosh HD']
    resultado=[]
    for p in candidatas:
        if not p.is_dir():continue
        p=p.resolve()
        if p not in resultado:resultado.append(p)
    # Si ya se recorre Documents, no repetir la subcarpeta del proyecto.
    return [p for p in resultado if not any(q!=p and q in p.parents for q in resultado)]


def semilla_argumentos(ruta):
    try:
        with zipfile.ZipFile(ruta) as z:
            entrada=next(n for n in z.namelist() if n.endswith('/data.pkl'))
            ops=list(pickletools.genops(z.read(entrada)))
        for i,(_,valor,_) in enumerate(ops):
            if valor!='seed':continue
            j=i+1
            while ops[j][0].name in ['MEMOIZE','BINPUT','LONG_BINPUT']:j+=1
            if ops[j][0].name in ['INT','BININT','BININT1','BININT2','LONG1','LONG4']:
                return int(ops[j][1])
    except (OSError,ValueError,KeyError,StopIteration,zipfile.BadZipFile):pass
    return None


def json_cercano(carpeta,nombre):
    for p in [carpeta,*list(carpeta.parents)[:3]]:
        q=p/nombre
        if q.is_file():
            try:return json.loads(q.read_text(encoding='utf-8-sig')),q
            except (OSError,ValueError):pass
    return {},None


def describir(ruta):
    meta,meta_path=json_cercano(ruta.parent,'metadatos.json')
    cfg,cfg_path=json_cercano(ruta.parent,'config.json')
    estado,estado_path=json_cercano(ruta.parent,'trainer_state.json')
    texto=(str(ruta)+' '+str(meta.get('modelo_base',''))).lower()
    arquitectura=cfg.get('model_type','')
    if 'robertuito' in texto:modelo='RoBERTuito'
    elif 'distilbeto' in texto or 'distilbert' in texto or arquitectura=='distilbert':modelo='DistilBERT'
    elif 'beto' in texto or arquitectura=='bert':modelo='BERT'
    else:modelo='No identificado'
    variante=meta.get('variante','')
    if variante not in VARIANTES:
        coincidencias=re.findall(r'(?:^|[/\\_.-])(mx|es|cu)(?=$|[/\\_.-])',str(ruta).lower())
        variante=coincidencias[-1] if coincidencias else ''
    semilla=meta.get('seed')
    origen='metadatos' if semilla is not None else ''
    if semilla is None:
        for p in [ruta.parent,*list(ruta.parent.parents)[:3]]:
            if (p/'training_args.bin').is_file():
                semilla=semilla_argumentos(p/'training_args.bin')
                if semilla is not None:origen='training_args';break
    if semilla is None:
        m=re.findall(r'(?:seed|semilla)[_-]?(\d+)',texto)
        if m:semilla=int(m[-1]);origen='nombre_de_ruta'
    contexto=('exportado_con_metadatos' if meta_path and meta_path.parent==ruta.parent else
        'checkpoint_test_final' if 'test_final' in texto else 'candidato_requiere_revision')
    return {'modelo':modelo,'variante':variante,'seed':semilla,'origen_seed':origen,
        'contexto':contexto,'ruta_pesos':str(ruta),'bytes':ruta.stat().st_size,
        'metadatos':str(meta_path or ''),'config':str(cfg_path or ''),
        'epoch_trainer_state':estado.get('epoch',''),
        'arquitectura':arquitectura,'config_id':meta.get('config_id','')}


def guardar_csv(ruta,filas,columnas):
    with Path(ruta).open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=columnas);w.writeheader();w.writerows(filas)


def localizar_pesos(raices,salida):
    salida=Path(salida).resolve();salida.mkdir(parents=True,exist_ok=True)
    encontrados=[];archivos=[];errores=[];visitados=set();n_carpetas=0
    for raiz in raices:
        print('Buscando en:',raiz,flush=True)
        def error(e):errores.append({'ruta':str(e.filename),'error':str(e)})
        for carpeta,subcarpetas,nombres in os.walk(raiz,topdown=True,onerror=error,followlinks=False):
            subcarpetas[:]=[n for n in subcarpetas if n not in OMITIR]
            n_carpetas+=1
            if n_carpetas%10000==0:print('Carpetas revisadas:',n_carpetas,flush=True)
            for nombre in nombres:
                p=Path(carpeta)/nombre
                if str(p) in visitados:continue
                visitados.add(str(p))
                low=nombre.lower()
                peso=(low.endswith('.safetensors') or
                      (low.endswith(('.bin','.pt','.pth','.ckpt')) and
                       any(t in low for t in ['model','beto','bert','weights','checkpoint'])))
                if peso:
                    try:encontrados.append(describir(p))
                    except (OSError,ValueError) as e:errores.append({'ruta':str(p),'error':str(e)})
                elif low.endswith(('.zip','.7z','.rar','.tar','.tar.gz')) and any(t in low for t in
                        ['sarcas','transform','beto','bert','model','checkpoint','respaldo','backup','resultados']):
                    archivos.append({'ruta':str(p),'estado':'respaldo_comprimido_posible; contenido no inspeccionado por este localizador'})
    campos=['modelo','variante','seed','origen_seed','contexto','ruta_pesos','bytes','metadatos','config',
            'epoch_trainer_state','arquitectura','config_id']
    guardar_csv(salida/'pesos_encontrados.csv',encontrados,campos)
    guardar_csv(salida/'respaldos_comprimidos.csv',archivos,['ruta','estado'])
    guardar_csv(salida/'carpetas_no_accesibles.csv',errores,['ruta','error'])
    cobertura=[]
    for modelo in MODELOS:
        for variante in VARIANTES:
            for semilla in SEMILLAS:
                candidatos=[x for x in encontrados if x['modelo']==modelo and x['variante']==variante and x['seed']==semilla]
                finales=[x for x in candidatos if x['contexto'] in ['exportado_con_metadatos','checkpoint_test_final']]
                cobertura.append({'modelo':modelo,'variante':variante,'seed':semilla,
                    'n_candidatos':len(candidatos),'n_con_contexto_final':len(finales),
                    'rutas':' | '.join(x['ruta_pesos'] for x in candidatos)})
    guardar_csv(salida/'cobertura_cinco_semillas.csv',cobertura,
                ['modelo','variante','seed','n_candidatos','n_con_contexto_final','rutas'])
    resumen={'raices':[str(p) for p in raices],'carpetas_revisadas':n_carpetas,
        'carpetas_no_accesibles':len(errores),'archivos_pesos':len(encontrados),
        'respaldos_comprimidos_posibles':len(archivos),
        'combinaciones_esperadas':45,
        'combinaciones_con_candidato':sum(x['n_candidatos']>0 for x in cobertura),
        'combinaciones_con_contexto_final':sum(x['n_con_contexto_final']>0 for x in cobertura),
        'nota':'Encontrar un checkpoint no demuestra que sea el refit final completo: revisar epoch, configuración y procedencia antes de usarlo.'}
    (salida/'resumen_localizacion.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
    texto=['# Localización de modelos Transformer','',
        f'Se encontraron {len(encontrados)} archivos de pesos. De las 45 combinaciones de modelo, variante y semilla, '
        f'{resumen["combinaciones_con_candidato"]} tienen candidatos y {resumen["combinaciones_con_contexto_final"]} tienen contexto de exportación o test final.','',
        '## Modelos identificados','']
    texto += [f'- {x["modelo"]} · {x["variante"]} · semilla {x["seed"]}: {x["ruta_pesos"]} ({x["contexto"]}).' for x in encontrados]
    texto += ['',f'Carpetas inaccesibles: {len(errores)}. Respaldos comprimidos posibles: {len(archivos)}.',
        '','Revisar cobertura_cinco_semillas.csv, pesos_encontrados.csv y respaldos_comprimidos.csv. '
        'Si no aparecen los modelos, no se concluye que nunca hayan existido: puede haber copias comprimidas o ubicaciones no accesibles. '
        'Este cuaderno solo localiza y describe archivos; no entrena, no carga los pesos, no elimina nada ni modifica las métricas.']
    (salida/'Informe_localizacion.md').write_text('\n'.join(texto),encoding='utf-8')
    print(json.dumps(resumen,ensure_ascii=False,indent=2));print('Informe en:',salida)
    return resumen


In [ ]:
RAICES = raices_automaticas(RAIZ_PROYECTO)
RESUMEN = localizar_pesos(RAICES, SALIDA)

## Resultado

El informe `Informe_localizacion.md` indica dónde están los candidatos. `cobertura_cinco_semillas.csv` muestra las 45 combinaciones esperadas. `pesos_encontrados.csv` conserva sus rutas, semillas y procedencia. Si aparecen respaldos comprimidos o carpetas inaccesibles, figuran en archivos separados.

Este cuaderno no reevalúa el test ni sustituye los resultados. Primero se debe confirmar que los candidatos corresponden a los entrenamientos finales y no a folds internos o checkpoints parciales.
